<a href="https://colab.research.google.com/github/sm-11/Github-Portfolio/blob/main/SelMe_zip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [168]:
# importing libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix

# 1. Regression Modelling

**Data Cleaning**

In [169]:
CRDC_df = pd.read_csv('ccd_devices.csv') # read csv file

# handle missing variables in the outcome variable
CRDC_df['SCH_INTERNET_WIFIENDEV'] = CRDC_df['SCH_INTERNET_WIFIENDEV'].replace(99, np.nan) # replacing '99' with NaN
CRDC_clean = CRDC_df.dropna(subset=['SCH_INTERNET_WIFIENDEV']) # dropping rows with NaN values

CRDC_clean.head(20)  # pring first 20 rows

,SCHID,SCH_NAME,SCH_UGDETAIL_ES,SCH_UGDETAIL_MS,SCH_UGDETAIL_HS,SCH_STATUS_SPED,SCH_STATUS_MAGNET,SCH_STATUS_CHARTER,SCH_STATUS_ALT,SCH_MAGNETDETAIL,SCH_INTERNET_WIFIENDEV
0,1063,Mesa Middle,False,False,False,False,True,False,False,False,0.0
1,1275,David A. Harrison Elem,False,False,False,False,False,False,False,False,1.0
2,2072,Lavergne Middle School,False,False,False,False,False,False,False,False,1.0
3,1862,Maurice River Township School,False,False,False,False,False,False,False,False,1.0
4,1533,Rosser Early Learning Center,False,False,False,False,False,False,False,False,0.0
5,5720,North Ridge El,False,False,False,False,False,False,False,False,1.0
6,4932,Bertha Neal School,False,False,False,False,False,False,False,False,1.0
7,1473,Pressley Ridge @ Grant Gardens,False,False,False,False,False,False,True,False,0.0
8,7666,Plum Creek El,False,False,False,False,False,False,False,False,1.0
9,7110,Newcomer Center,False,False,False,False,False,False,True,False,1.0


A) How many schools report having enough WiFi-enabled devices and how many do not?

In [170]:
# counting the frequency of uinque values
device_count = CRDC_clean['SCH_INTERNET_WIFIENDEV'].value_counts()
total_schools = device_count.sum() # total num. of schools

# assigning variables to the unique values
enough = device_count.get(1,0)
not_enough = device_count.get(0,0)

# calculating in percentages
percent_enough = enough / total_schools * 100
percent_notenough = not_enough / total_schools * 100

# printing values
print(f"Number of schools with enough WiFi-enabled devices: {enough} or {percent_enough.round(2)}%")
print(f"Number of schools without enough WiFi-enabled devices: {not_enough} or {percent_notenough.round(2)}%")

Number of schools with enough WiFi-enabled devices: 4029 or 81.25%
Number of schools without enough WiFi-enabled devices: 930 or 18.75%


Out of the 4959 schools, 81.25% of schools report having enough WiFi-enabled devices, while 18.75% of schools report not having enough devices.




**Predictive Modeling**

B) Make a simple regression model predicting SCH_INTERNET_WIFIENDEV using one school
characteristic of your choice. Values of 99 indicate missing data and should be handled
appropriately. (you do not need to test regression assumptions).  

In [171]:
# define X and y for training and testing model
X = CRDC_clean[['SCH_STATUS_MAGNET']]
y = CRDC_clean['SCH_INTERNET_WIFIENDEV']

# split train/test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# fit model
log_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = log_model.predict(X_test)

# evaluate performance of model
accuracy = log_model.score(X_test, y_test)
precision = precision_score(y_test, y_pred)
coef = log_model.coef_[0][0]

# print outcomes
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Coefficient: {coef:.3f}")

Accuracy: 0.823
Precision: 0.823
Coefficient: 0.055


C) Briefly describe the results, focusing on the relationship between the predictor you chose and
device availability (2-3 sentences).

Having a Magnet school program, or being a Magnet school, is associated with a very small positive effect on the availability of WiFi-enabled devices in schools. The logistic regression model shows that the presence of a Magnet program increases the odds of a school having WiFi-enabled devices by only 5.5%. While this effect is modest, the model itself demonstrates reasonably good predictive performance, correctly classifying schools about 82% of the time.

D) Modify your model or use a new one to improve performance. We are only looking for any
improvement over the original model, not perfect performance. Briefly describe what you did
and the results.

In [165]:
# employing a decision tree model
# fit and predict
tree_model = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=1).fit(X_train, y_train) # balanced weight because the model was too biased in predicting 'enough wifi' schools
y_pred = tree_model.predict(X_test)

# evaluate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# print out results
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.200
Precision: 0.827
Confusion Matrix:
[[ 254    9]
 [1182   43]]


To improve model performance, a Decision Tree was initially employed. However, with the default settings, the model performed poorly: it was biased toward predicting only the majority class, "schools with enough WiFi", and completely failed to detect schools with insufficient WiFi-enabled devices. To address this, we applied class_weight='balanced' to give importance to the minority class. This adjustment successfully improved detection of the “not enough devices” schools, as seen in the confusion matrix, but it skewed the model toward the minority class, drastically reducing overall accuracy from 82% to 20%, while precision for the majority class remained relatively stable. This demonstrates that the Decision Tree can detect minority class instances better with class balancing, but with only one weak predictor, the model still struggles to achieve overall reliable predictions.

Therefore, logistic regression is better suited for this specific predictor because the predictor has a weak, likely non-linear relationship with the outcome. Performance of the models could be improved by:


*   Adding more predictive variables to give the model more information

*   Hyperparameter tuning (e.g., adjusting max_depth, min_samples_split, or criterion in a Decision Tree)
*   Using alternative evaluation metrics (precision, recall, F1-score, ROC-AUC) to better interpret model performance beyond accuracy.


Overall, with the current single predictor, logistic regression provides the most reliable and interpretable results.

# 2. Implementing an Edit-Distance Algorithm

In [195]:
def hamming_distance(word1, word2):

    distance = 0 # initialize mismatch counter
    max_len = max(len(word1), len(word2)) # determine max length of the two words

    for i in range(max_len):
        # if index exists in both words, compare
        if i < len(word1) and i < len(word2):
            if word1[i] != word2[i]:
                distance += 1
        else:
            # extra characters count as mismatches
            distance += 1

    return distance

# Test the examples
print("Distance between 'potting' and 'putting':", hamming_distance("potting", "putting"))
print("Distance between 'banana' and 'bananas':", hamming_distance("banana", "bananas"))
print("Distance between 'cake' and 'baked':", hamming_distance("cake", "baked"))
print("Distance between 'talking' and 'pickings':", hamming_distance("talking", "pickings"))

Distance between 'potting' and 'putting': 1
Distance between 'banana' and 'bananas': 1
Distance between 'cake' and 'baked': 2
Distance between 'talking' and 'pickings': 4


In [196]:
def phrase_distance(phrase1, phrase2):
  # split strings to substrings
    words1 = phrase1.split()
    words2 = phrase2.split()

    # compare word by word
    distance = 0 # initialize mismatch counter
    min_len = min(len(words1), len(words2)) # determine max length of the two words

    for i in range(min_len):
        distance += hamming_distance(words1[i], words2[i])

    # add extra words as mismatches
    for extra_word in words1[min_len:]:
        distance += len(extra_word)
    for extra_word in words2[min_len:]:
        distance += len(extra_word)

    return distance

# Test the examples
print("Distance between 'two dogs and two cats' and 'two cats and two dogs':",
      phrase_distance("two dogs and two cats", "two cats and two dogs"))

print("Distance between 'two dogs and two cats' and 'three dogs and two cats':",
      phrase_distance("two dogs and two cats", "three dogs and two cats"))

Distance between 'two dogs and two cats' and 'two cats and two dogs': 6
Distance between 'two dogs and two cats' and 'three dogs and two cats': 4
